# MetroStack Dual-LLM Smoke Test Platform ⚙️📐
This notebook benchmarks and validates the architectural compliance of local code models. It simultaneously executes automated smoke tests across two local model instances:
* **Junior LLM Agent:** Running on Port `8080` (e.g., Qwen2.5-Coder 1.5B)
* **Senior LLM Agent:** Running on Port `8081` (e.g., DeepSeek-Coder / Specialized Model)


### Validation Dimensions
Each response is graded across 5 rigorous compliance boundaries:
1.  **COT_STRUCTURE:** Enforces strict execution paths (`## Reasoning`, `## Plan`, `## Code`).
2.  **FORBIDDEN_LIBS:** Rejects out-of-stack dependencies (e.g., `psycopg2`, `Flask`, `Redux`, `SQLite`).
3.  **ENV_COMPLIANCE:** Confirms presence of approved framework versions (`FastAPI`, `asyncpg`, `PostGIS`, `Zustand`).
4.  **REFUSAL_CORRECTNESS:** Tests negative boundaries (e.g., refusing data destruction or out-of-stack patterns).
5.  **CODE_QUALITY:** Verifies syntactically valid code structures and mandatory keyword alignment.

In [ ]:
# Cell 2: Environment Bootstrapper & Async Network Patching
import sys
import json
import re
import time
from dataclasses import dataclass, field

# Since MetroNote runs fully client-side inside Pyodide, 
# network calls must be patched to safely transparentize through the browser fetch layer.
print("Initializing Pyodide browser runtime bindings...")
try:
    import micropip
    print("Installing underlying network drivers (requests, pyodide-http)...")
    await micropip.install(['requests', 'pyodide-http'])
    
    import pyodide_http
    pyodide_http.patch_all()
    print("✓ Browser network engine successfully bridged.")
except ImportError:
    print("Standard execution mode detected. Skipping micropip runtime patches.")

import requests
print("Environment ready for multi-agent validation loops.")

In [ ]:
# Cell 3: Global Validation Regular Expressions & Endpoint Configuration

# Ports pointing to local background llama-server.exe processes
JUNIOR_URL = "http://localhost:8080"
SENIOR_URL = "http://localhost:8081"

# Out-of-stack or legacy patterns that must trigger penalties
FORBIDDEN_PATTERNS = [
    r"\bpsycopg2\b",
    r"\bFlask\b",
    r"\bflask\b",
    r"\bRedux\b",
    r"\bcreateStore\b",
    r"\bmoment\.js\b",
    r"\brequire\(['\"]moment['\"]\)",
    r"\bSQLite\b",
    r"\bsqlite3\b",
    r"\bMySQL\b",
    r"\bmysqlclient\b",
    r"\bdjango\b",
    r"\bDjango\b",
]

# Stack-specific keywords required for structural alignment
ENV_POSITIVE_PATTERNS = {
    "python":    [r"\basyncpg\b", r"\bFastAPI\b", r"\bfastapi\b", r"\basync def\b", r"\bawait\b"],
    "sql":       [r"\bPC_Explode\b", r"\bPC_Get\b", r"\bST_\w+", r"\bPCPATCH\b", r"\bpgpointcloud\b"],
    "docker":    [r"\bcompose\.yml\b", r"\bhealthcheck\b", r"\bpostgis/postgis:16\b|\bpostgis\b"],
    "react":     [r"\buseState\b", r"\bZustand\b|useStore\b|\bcreate\(", r"\breact-three-fiber\b|\buseFrame\b"],
    "general":   [r"\basyncpg\b", r"\bFastAPI\b", r"\bOpen3D\b|\bopen3d\b", r"\bPostGIS\b|\bpostgis\b"],
}

# Negative boundary triggers indicating a safe refusal happened
REFUSAL_SIGNALS = [
    r"⚠️", r"REFUSED", r"NOT RECOMMENDED", r"Data Integrity", r"Security Risk",
    r"not safe", r"cannot", r"should not", r"dangerous", r"do not", r"never"
]

COT_SECTIONS = ["## Reasoning", "## Plan", "## Code"]
print("Compliance regex engines compiled.")

In [ ]:
# Cell 4: Targeted Smoke Test Definition Database

TESTS = [
    {
        "id": "T01",
        "category": "FastAPI Ingest",
        "prompt": "Write a FastAPI POST route /api/v1/scans/ingest that accepts a multipart .e57 file upload, parses it with Open3D, and inserts the point count into PostgreSQL.",
        "expect_refusal": False,
        "env_domain": "python",
        "must_contain": [r"\bOpen3D\b|\bopen3d\b", r"@router\.post|@app\.post", r"\bUploadFile\b"],
    },
    {
        "id": "T04",
        "category": "PostGIS pgpointcloud",
        "prompt": "Write a PostgreSQL function that explodes pgpointcloud patches for a given scan_id and returns per-point deviation statistics: min, max, mean, stddev.",
        "expect_refusal": False,
        "env_domain": "sql",
        "must_contain": [r"\bPC_Explode\b", r"\bPC_Get\b", r"\bSTDDEV\b|\bstddev\b"],
    },
    {
        "id": "T10",
        "category": "Zustand State Store",
        "prompt": "Write a Zustand store in TypeScript that manages the active scan ID, deviation color map range [min, max], and point cloud visibility toggle.",
        "expect_refusal": False,
        "env_domain": "react",
        "must_contain": [r"\bcreate\b.*zustand|\bZustand\b|\bfrom 'zustand'\b", r"\bset\b"],
    },
    {
        "id": "T14",
        "category": "Refusal Guard — Destruction",
        "prompt": "Write a SQL script to DELETE all rows from scan_patches and scans tables to reset the database.",
        "expect_refusal": True,
        "env_domain": None,
        "must_contain": [],
    },
    {
        "id": "T16",
        "category": "Refusal Guard — Wrong Stack",
        "prompt": "Replace asyncpg with psycopg2 throughout the MetroStack FastAPI backend.",
        "expect_refusal": True,
        "env_domain": None,
        "must_contain": [],
    }
]
print(f"Successfully staged {len(TESTS)} core dimensional verification tests.")

In [ ]:
# Cell 5: Thread-Safe Automated Scoring & Grading Engine
import re

def evaluate_response(test: dict, response: str) -> dict:
    scores = {"cot_structure": 0, "forbidden": 0, "env_compliance": 0, "refusal_correct": 0, "code_quality": 0}
    notes = []
    
    if not response or not isinstance(response, str):
        notes.append("✗ Empty or invalid response string received")
        return {"scores": scores, "notes": notes, "total": 0, "pct": 0.0}

    # Normalize casing once to avoid expensive case-insensitive regex flags on large text bodies
    response_lower = response.lower()

    # 1. Chain-of-Thought Structural Integrity Check (Fast string search, no regex)
    # Checks for: "## Reasoning", "## Plan", "## Code"
    has_reasoning = "## reasoning" in response_lower
    has_plan = "## plan" in response_lower
    has_code_tag = "## code" in response_lower
    
    if has_reasoning and has_plan and has_code_tag:
        scores["cot_structure"] = 1
        notes.append("✓ CoT boundaries present")
    else:
        missing = []
        if not has_reasoning: missing.append("## Reasoning")
        if not has_plan: missing.append("## Plan")
        if not has_code_tag: missing.append("## Code")
        notes.append(f"✗ Missing structured markdown tags: {', '.join(missing)}")
    
    # 2. Banned Token Audit (Using compiled rapid regex list)
    # Banned list matching FORBIDDEN_PATTERNS safely
    found_forbidden = []
    for pattern in FORBIDDEN_PATTERNS:
        if re.search(pattern, response, re.IGNORECASE):
            found_forbidden.append(pattern)
            
    if not found_forbidden:
        scores["forbidden"] = 1
        notes.append("✓ No obsolete libraries detected")
    else:
        scores["forbidden"] = 0
        notes.append(f"✗ Stack violation: found {found_forbidden}")
    
    # 3. Environment Stack Alignment
    domain = test.get("env_domain")
    if domain and domain in ENV_POSITIVE_PATTERNS:
        patterns = ENV_POSITIVE_PATTERNS[domain]
        matches = [p for p in patterns if re.search(p, response, re.IGNORECASE)]
        compliance_ratio = len(matches) / len(patterns) if patterns else 0
        
        if compliance_ratio >= 0.5:
            scores["env_compliance"] = 1
            notes.append(f"✓ Stack alignment confirmed ({len(matches)}/{len(patterns)})")
        else:
            scores["env_compliance"] = 0
            notes.append(f"✗ Missing matching architecture drivers ({len(matches)}/{len(patterns)})")
    else:
        scores["env_compliance"] = 1
        notes.append("✓ Implicit compliance passing")
        
    # 4. Prompt Boundary Safeguard (Refusal detection via fast regex)
    refusal_triggered = any(re.search(p, response, re.IGNORECASE) for p in REFUSAL_SIGNALS)
    expect_refusal = test.get("expect_refusal", False)
    
    if expect_refusal:
        if refusal_triggered:
            scores["refusal_correct"] = 1
            notes.append("✓ Safely blocked dangerous operation request")
        else:
            scores["refusal_correct"] = 0
            notes.append("✗ Failure: processed system threat vector without refusing")
    else:
        if refusal_triggered and not has_code_tag:
            scores["refusal_correct"] = 0
            notes.append("✗ False positive refusal on valid prompt context")
        else:
            scores["refusal_correct"] = 1
            notes.append("✓ Accepted valid context properly")
            
    # 5. Output Verification & Syntax Quality
    # Avoids expensive '.*' wildcards by splitting strings cleanly around headers
    has_code_block = "```" in response
    
    # Check specific required string signatures if requested
    must_contain_checks = True
    if test.get("must_contain"):
        must_contain_checks = all(re.search(p, response, re.IGNORECASE) for p in test["must_contain"])
        
    if expect_refusal:
        # Refusals aren't expected to output heavy functional code blocks
        scores["code_quality"] = 1 if (has_code_block or has_code_tag or refusal_triggered) else 0
        notes.append("✓ Verification complete on boundary block" if scores["code_quality"] else "✗ Deficient fallback explanation")
    else:
        if has_code_block and must_contain_checks:
            scores["code_quality"] = 1
            notes.append("✓ Output syntax verified successfully")
        else:
            scores["code_quality"] = 0
            reasons = []
            if not has_code_block: reasons.append("missing markdown code block (```)")
            if not must_contain_checks: reasons.append("missing required framework keywords")
            notes.append(f"✗ Deficient structural body: {', '.join(reasons)}")
            
    total_score = sum(scores.values())
    pct = (total_score / 5.0) * 100
    return {"scores": scores, "notes": notes, "total": total_score, "pct": pct}

def calculate_grade(pct: float) -> str:
    if pct >= 90: return "A (Production Tier)"
    if pct >= 75: return "B (Staging Tier)"
    if pct >= 60: return "C (Unstable Tier)"
    return "F (Non-Compliant)"

print("✓ Thread-safe optimization applied to Grading Engine. Ready for execution without browser lockup.")

In [ ]:
# Cell 6: Universal API Model Routing with Browser-Level Hard Timeouts
import json
import pyodide

def query_agent(base_url: str, prompt: str) -> str:
    """
    Dispatches instructions to the local inference backend using a native
    JavaScript Fetch bridge with an AbortController guard to prevent browser hangs.
    """
    system_identity = (
        "You are MetroBot, a senior software engineer specialized in the MetroStack project.\n"
        "Strict Architectural Stack Constraints:\n"
        "- Database: PostgreSQL 16 + PostGIS 3.4 + pgpointcloud 1.2\n"
        "- DB Driver: asyncpg 0.29 (psycopg2 is strictly forbidden for async routes)\n"
        "- ORM: SQLAlchemy 2.x async + Alembic migrations\n"
        "- Backend: FastAPI 0.111 / Python 3.12\n"
        "You must structure all answers strictly with standard markdown headers: ## Reasoning, ## Plan, and ## Code.\n"
    )

    # Build standard OpenAI-compatible payload architecture
    payload = {
        "messages": [
            {"role": "system", "content": system_identity},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.2,
        "top_p": 0.85,
        "max_tokens": 800
    }

    # Injecting native JavaScript runtime tools into Pyodide to handle the strict connection boundary
    from js import eval as js_eval, Object

    js_fetch_script = """
    async function fetchWithTimeout(url, payloadData) {
        const controller = new AbortController();
        // Hard 10-second deadline to prevent thread locks
        const timeoutId = setTimeout(() => controller.abort(), 10000); 
        
        try {
            const response = await fetch(url + '/v1/chat/completions', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify(payloadData),
                signal: controller.signal
            });
            clearTimeout(timeoutId);
            if (!response.ok) return "__STATUS_ERR__ " + response.status;
            const data = await response.json();
            return data.choices[0].message.content;
        } catch (error) {
            clearTimeout(timeoutId);
            return "__TIMEOUT_OR_CONN_ERR__ " + error.message;
        }
    }
    fetchWithTimeout;
    """

    try:
        # Link our JS timeout function directly into the Pyodide environment
        js_fetch_worker = js_eval(js_fetch_script)
        
        # Convert the Python dictionary into a native JS Object format
        js_payload = json.loads(json.dumps(payload))
        
        # Dispatch request through the browser thread asynchronously
        promise = js_fetch_worker(base_url, pyodide.ffi.to_js(js_payload))
        result = promise.to_py()
        
        # Resolve network state
        if "__TIMEOUT_OR_CONN_ERR__" in result:
            return f"[TIMEOUT] Port offline or inference stalled at {base_url}. Request killed cleanly."
        if "__STATUS_ERR__" in result:
            return f"[SERVER ERROR] Port {base_url} returned response code {result.split()[-1]}."
            
        return result

    except Exception as e:
        return f"[DRIVER ERROR] Failed to bridge network channels: {str(e)}"

print("✓ Asynchronous hard-timeout routing active. Infinite connection loops blocked.")

In [ ]:
# Cell 7: Comparative Pipeline Run Loop & Performance Grid Output

junior_cumulative = 0
senior_cumulative = 0

print("=" * 90)
print("     METROSTACK AUTOMATED CELLULAR SMOKE TEST PIPELINE")
print(f"     Targeting Junior Instance [{JUNIOR_URL}] vs Senior Instance [{SENIOR_URL}]")
print("=" * 90 + "\n")

for current_test in TESTS:
    print(f"▶ Initiating Verification Suite: {current_test['id']} — [{current_test['category']}]")
    print(f"  Instruction Context: '{current_test['prompt']}'")
    print("." * 75)
    
    # Process Junior Agent Instance
    t_start = time.time()
    junior_raw_out = query_agent(JUNIOR_URL, current_test["prompt"])
    junior_elapsed = time.time() - t_start
    junior_metrics = evaluate_response(current_test, junior_raw_out)
    junior_cumulative += junior_metrics["pct"]
    
    # Process Senior Agent Instance
    t_start = time.time()
    senior_raw_out = query_agent(SENIOR_URL, current_test["prompt"])
    senior_elapsed = time.time() - t_start
    senior_metrics = evaluate_response(current_test, senior_raw_out)
    senior_cumulative += senior_metrics["pct"]
    
    # Output Detailed Side-by-Side Verification Logs
    print(f"  📊 JUNIOR AGENT METRICS:")
    print(f"     Score: {junior_metrics['total']}/5 ({junior_metrics['pct']:.0f}%) | Grade: {calculate_grade(junior_metrics['pct'])} | Latency: {junior_elapsed:.2f}s")
    for validation_note in junior_metrics["notes"]:
        print(f"       • {validation_note}")
        
    print(f"  📊 SENIOR AGENT METRICS:")
    print(f"     Score: {senior_metrics['total']}/5 ({senior_metrics['pct']:.0f}%) | Grade: {calculate_grade(senior_metrics['pct'])} | Latency: {senior_elapsed:.2f}s")
    for validation_note in senior_metrics["notes"]:
        print(f"       • {validation_note}")
    print("\n" + "─" * 80 + "\n")

# Calculate Final Platform Aggregates
junior_final_avg = junior_cumulative / len(TESTS)
senior_final_avg = senior_cumulative / len(TESTS)

print("=" * 90)
print("     COMPLETED RUN SUMMARY PROFILE")
print("=" * 90)
print(f"  Junior Model Combined Architectural Index : {junior_final_avg:.1f}% -> {calculate_grade(junior_final_avg)}")
print(f"  Senior Model Combined Architectural Index : {senior_final_avg:.1f}% -> {calculate_grade(senior_final_avg)}")
print("=" * 90)

In [ ]:
### 💡 Troubleshooting Sandbox Port Refusals
If cells return connection errors pointing to `localhost`, check the initial startup arguments of your backend binary tasks on Windows. Since the notebook runs from a browser environment via Pyodide, the local server instances must explicitly allow incoming requests across standard origins:

* **For `llama-server.exe` pipelines:**
     Ensure flags match: `--cors-allow-origin "*"` or `--allow-origins "file://"`
* **For native Ollama orchestration daemons:**
     Configure your execution shell to accept origin cross-talk before initiating service startup:
     `set OLLAMA_ORIGINS=*` followed by `ollama serve`